# Evaluation 

Dataset: MUSTARD 

Input: LLEasAttention based representations

Model: MLP

Output: binary classification results

# Baselines

## (U) Unimodal into MLP, considering number of votes $V$

In [16]:
import os
import pandas as pd
from sklearn.metrics import classification_report

from evaluate_with_MLP import get_predictions, evaluate_labels, extract_results_from_report

def evaluate_votes_MLP(_dir_file_pkl, _str_dataset, _str_video_representation, _variation, _m, _plot, _DIR_RESULTS, _votes):
    """
    Params
    ------
    _dir_file_pkl: str, pkl file path that contains the representations
    _str_dataset: str, options: {mustard}
    _str_video_representations, str, options: {promSumVectors, bagVectors}
    _m: str, modality between T, V and A
    _plot: bool
    _DIR_RESULTS: str, results will be saved here
    _votes: int, votes threshold
    """
    os.makedirs(_DIR_RESULTS, exist_ok=True)

    if _str_dataset == "mustard":
        _n = 50 # value by default
    
    print(_dir_file_pkl)
    name = _dir_file_pkl.split("/")[-1][:-4]
    y_test_M, lst_votes_M = get_predictions(_dir_file_pkl, _str_dataset, _str_video_representation, _m, _plot)
    print("Finished get_predictions")
    
    # Save votes results 
    votes = pd.DataFrame()
    votes[_m] = lst_votes_M
    csv_votes = _DIR_RESULTS + f"votes_{name}.csv"
    votes.to_csv(csv_votes, header="sarcasm", index=False)
    print("Saved", csv_votes)

    print("\nVotes threshold:", _votes)
    y_test_M, arr_preds_M = evaluate_labels(_votes, y_test_M, lst_votes_M, _n)
    print("Finished evaluate_labels")
    report_M = classification_report(y_pred=arr_preds_M, y_true=y_test_M , digits=4, output_dict=True)
    print("Finished classification_report")
    
    # Save prediction results
    preds = pd.DataFrame()
    preds[_m] = arr_preds_M
    csv_preds = _DIR_RESULTS + f"preds_v{_votes}_{name}.csv"
    preds.to_csv(csv_preds, header=f"{_variation}_{_m}", index=False)
    print("Saved", csv_preds)

    # Evaluation results to txt files
    file_txt = _DIR_RESULTS + f"results_{_variation}.txt"
    extract_results_from_report(name, _votes, _m, report_M, file_txt)

In [17]:
# MUSTARD DATASET, Unimodal (U) representations

dir_file_pkl = "../data/sarcasm.pkl" 
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "U" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "T"
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 10

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

../data/sarcasm.pkl
File:'../data/sarcasm.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: T.
CLF: 73 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_sarcasm.csv

Votes threshold: 10
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v10_sarcasm.csv
votes: 10, modality: T, acc: 0.6304347826086957, macro-f1: 0.5864621893178212, f1_0: 0.7213114754098361, f1_1: 0.45161290322580644, variation: sarcasm

Results saved on: ./eval_MLP_votes/results_U.txt


## (CM) $M_1$ + $M_2$ into MLP, considering number of votes $V$ 

In [18]:
import os
import pandas as pd
from sklearn.metrics import classification_report

from evaluate_with_MLP import get_predictions_2modalities, evaluate_labels, extract_results_from_report

def evaluate_concat2m_votes_MLP(_dir_file_pkl1, _dir_file_pkl2, _str_dataset, _str_video_representation, _variation, _m1, _m2, _DIR_RESULTS, _votes):
    os.makedirs(_DIR_RESULTS, exist_ok=True)

    if _str_dataset == "mustard":
        _n = 50 # by default
    
    name1 = _dir_file_pkl1.split("/")[-1][:-4]
    name2 = _dir_file_pkl2.split("/")[-1][:-4]

    y_test, lst_votes = get_predictions_2modalities(_dir_file_pkl1, _dir_file_pkl2, 
                                                    _str_dataset, _str_video_representation, 
                                                    _m1, _m2)
    print("Finished get_predictions_2modalities")
    
    _m = f"{_m1}+{_m2}"
    name = f"{name1}+{name2}"
    
    votes = pd.DataFrame()
    votes[_m] = lst_votes
    csv_votes = _DIR_RESULTS + f"votes_{_variation}.csv"
    votes.to_csv(csv_votes, header="sarcasm", index=False)
    print("Saved", csv_votes)

    print("\nVotes threshold:", _votes)
    y_test_M, arr_preds_M = evaluate_labels(_votes, y_test, lst_votes, _n)
    print("Finished evaluate_labels")
    report_M = classification_report(y_pred=arr_preds_M, y_true=y_test_M , digits=4, output_dict=True)
    print("Finished classification_report")
    
    # Save prediction results
    preds = pd.DataFrame()
    preds[_m] = arr_preds_M
    csv_preds = _DIR_RESULTS + f"preds_v{_votes}_{_variation}.csv"
    preds.to_csv(csv_preds, header=f"{_variation}", index=False)
    print("Saved", csv_preds)

    # Evaluation results to txt files
    file_txt = _DIR_RESULTS + f"results_{_variation}.txt"
    #print(file_txt)
    extract_results_from_report(name, _votes, _m, report_M, file_txt)

In [19]:
dir_file_pkl1 = "../data/sarcasm.pkl" 
dir_file_pkl2 = "../data/sarcasm.pkl" 
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "CM" 

DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_concat2m_votes_MLP(dir_file_pkl1, dir_file_pkl2, str_dataset, str_video_representation,
                        variation, "T", "V",
                        DIR_RESULTS, votes)

Working on: mustard
Video representation: bagVectors
Working on: mustard
Video representation: bagVectors
CLF: 63 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions_2modalities
Saved ./eval_MLP_votes/votes_CM.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_CM.csv
votes: 6, modality: T+V, acc: 0.6304347826086957, macro-f1: 0.6260162601626016, f1_0: 0.6666666666666666, f1_1: 0.5853658536585366, variation: sarcasm+sarcasm

Results saved on: ./eval_MLP_votes/results_CM.txt


## (MM) $M_1$ + $M_2$ + $M_3$ into MLP, considering number of votes $V$ 

In [20]:
import os
import pandas as pd
from sklearn.metrics import classification_report

from evaluate_with_MLP import get_predictions_3modalities, evaluate_labels, extract_results_from_report

def evaluate_concat3m_votes_MLP(_dir_file_pkl1, _dir_file_pkl2, _dir_file_pkl3, _str_dataset, _str_video_representation, _variation, _m1, _m2, _m3, _DIR_RESULTS, _votes):
    os.makedirs(_DIR_RESULTS, exist_ok=True)

    if _str_dataset == "mustard":
        _n = 50 # by default
    
    name1 = _dir_file_pkl1.split("/")[-1][:-4]
    name2 = _dir_file_pkl2.split("/")[-1][:-4]
    name3 = _dir_file_pkl3.split("/")[-1][:-4]

    y_test, lst_votes = get_predictions_3modalities(_dir_file_pkl1, _dir_file_pkl2, _dir_file_pkl3,
                                                    _str_dataset, _str_video_representation, 
                                                    _m1, _m2, _m3)
    print("Finished get_predictions_2modalities")
    
    _m = f"{_m1}+{_m2}+{_m3}"
    name = f"{name1}+{name2}+{name3}"
    
    votes = pd.DataFrame()
    votes[_m] = lst_votes
    csv_votes = _DIR_RESULTS + f"votes_{_variation}.csv"
    votes.to_csv(csv_votes, header="sarcasm", index=False)
    print("Saved", csv_votes)

    print("\nVotes threshold:", _votes)
    y_test_M, arr_preds_M = evaluate_labels(_votes, y_test, lst_votes, _n)
    print("Finished evaluate_labels")
    report_M = classification_report(y_pred=arr_preds_M, y_true=y_test_M , digits=4, output_dict=True)
    print("Finished classification_report")
    
    # Save prediction results
    preds = pd.DataFrame()
    preds[_m] = arr_preds_M
    csv_preds = _DIR_RESULTS + f"preds_v{_votes}_{_variation}.csv"
    preds.to_csv(csv_preds, header=f"{_variation}", index=False)
    print("Saved", csv_preds)

    # Evaluation results to txt files
    file_txt = _DIR_RESULTS + f"results_{_variation}.txt"
    #print(file_txt)
    extract_results_from_report(name, _votes, _m, report_M, file_txt)

In [21]:
dir_file_pkl1 = "../data/sarcasm.pkl" 
dir_file_pkl2 = "../data/sarcasm.pkl"
dir_file_pkl3 = "../data/sarcasm.pkl" 
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "MM"  

DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_concat3m_votes_MLP(dir_file_pkl1, dir_file_pkl2, dir_file_pkl3,
                            str_dataset, str_video_representation,
                            variation, "T", "V", "A",
                            DIR_RESULTS, votes)

Working on: mustard
Video representation: bagVectors
Working on: mustard
Video representation: bagVectors
Working on: mustard
Video representation: bagVectors
CLF: 62 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions_2modalities
Saved ./eval_MLP_votes/votes_MM.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_MM.csv
votes: 6, modality: T+V+A, acc: 0.5942028985507246, macro-f1: 0.5928345626975764, f1_0: 0.6164383561643836, f1_1: 0.5692307692307692, variation: sarcasm+sarcasm+sarcasm

Results saved on: ./eval_MLP_votes/results_MM.txt


# Multimodal LLE-based Representations into MLP, with votes $V$

## (UM_N) Unimodal reconstructed into MLP, considering number of votes $V$

In [22]:
# MUSTARD DATASET, Unimodal (UM_N) representations

dir_file_pkl = "./LLEasAtt_representations/mustard_bagVectors_UM_N_10_T_V_A.pkl"
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "UM_N" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "T"
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

./LLEasAtt_representations/mustard_bagVectors_UM_N_10_T_V_A.pkl
File:'./LLEasAtt_representations/mustard_bagVectors_UM_N_10_T_V_A.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: T.
CLF: 66 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_mustard_bagVectors_UM_N_10_T_V_A.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_mustard_bagVectors_UM_N_10_T_V_A.csv
votes: 6, modality: T, acc: 0.6376811594202898, macro-f1: 0.6349206349206349, f1_0: 0.6031746031746031, f1_1: 0.6666666666666666, variation: mustard_bagVectors_UM_N_10_T_V_A

Results saved on: ./eval_MLP_votes/results_UM_N.txt


## Cross-modal (CM_NM or CM_NE) into MLP, considering number of votes $V$

### CM_NM

In [23]:
# MUSTARD DATASET, Cross-modal (CM_NM OR CM_NE) representations


dir_file_pkl = "./LLEasAtt_representations/mustard_bagVectors_CM_NM_7_7_TV_VT_AV.pkl"
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "CM_NM" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "T" # T<-V
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

./LLEasAtt_representations/mustard_bagVectors_CM_NM_7_7_TV_VT_AV.pkl
File:'./LLEasAtt_representations/mustard_bagVectors_CM_NM_7_7_TV_VT_AV.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: T.
CLF: 32 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_mustard_bagVectors_CM_NM_7_7_TV_VT_AV.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_mustard_bagVectors_CM_NM_7_7_TV_VT_AV.csv
votes: 6, modality: T, acc: 0.6521739130434783, macro-f1: 0.6485568760611206, f1_0: 0.6129032258064516, f1_1: 0.6842105263157895, variation: mustard_bagVectors_CM_NM_7_7_TV_VT_AV

Results saved on: ./eval_MLP_votes/results_CM_NM.txt


### CM_NE

In [24]:
dir_file_pkl = "./LLEasAtt_representations/mustard_bagVectors_CM_NE_5_5_TV_VT_AV.pkl"
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "CM_NE" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "V" # V<--T
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

./LLEasAtt_representations/mustard_bagVectors_CM_NE_5_5_TV_VT_AV.pkl
File:'./LLEasAtt_representations/mustard_bagVectors_CM_NE_5_5_TV_VT_AV.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: V.
CLF: 471 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_mustard_bagVectors_CM_NE_5_5_TV_VT_AV.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_mustard_bagVectors_CM_NE_5_5_TV_VT_AV.csv
votes: 6, modality: V, acc: 0.6159420289855072, macro-f1: 0.6049689440993788, f1_0: 0.6708074534161491, f1_1: 0.5391304347826087, variation: mustard_bagVectors_CM_NE_5_5_TV_VT_AV

Results saved on: ./eval_MLP_votes/results_CM_NE.txt


## Multi-modal (MM_NM or CM_NE) into MLP, considering number of votes $V$

### MM_NM

In [25]:
# MUSTARD DATASET, Multi-modal (MM_NM OR CM_NE) representations

dir_file_pkl = "./LLEasAtt_representations/mustard_bagVectors_MM_NM_15_TVA_VTA_AVT.pkl"
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "MM_NM" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "V" # V <- T,A
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 12

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

./LLEasAtt_representations/mustard_bagVectors_MM_NM_15_TVA_VTA_AVT.pkl
File:'./LLEasAtt_representations/mustard_bagVectors_MM_NM_15_TVA_VTA_AVT.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: V.
CLF: 509 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_mustard_bagVectors_MM_NM_15_TVA_VTA_AVT.csv

Votes threshold: 12
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v12_mustard_bagVectors_MM_NM_15_TVA_VTA_AVT.csv
votes: 12, modality: V, acc: 0.644927536231884, macro-f1: 0.6105511720324829, f1_0: 0.7262569832402235, f1_1: 0.4948453608247423, variation: mustard_bagVectors_MM_NM_15_TVA_VTA_AVT

Results saved on: ./eval_MLP_votes/results_MM_NM.txt


### MM_NE

In [26]:
# MUSTARD DATASET, Multi-modal (MM_NM OR CM_NE) representations

dir_file_pkl = "./LLEasAtt_representations/mustard_bagVectors_MM_NE_12_TVA_VTA_AVT.pkl"
# parameters
str_dataset = "mustard"
str_video_representation = "bagVectors" # promSumVectors OR bagVectors, according with your file
variation = "MM_NE" # variations: {U, UM_N, CM_NM, CM_NE, MM_NM, MM_NE}
m = "T" # T <-- V,A
plot = False
DIR_RESULTS = "./eval_MLP_votes/"
votes = 6

evaluate_votes_MLP(dir_file_pkl, str_dataset, str_video_representation,
                   variation, m, plot, 
                   DIR_RESULTS, votes)

./LLEasAtt_representations/mustard_bagVectors_MM_NE_12_TVA_VTA_AVT.pkl
File:'./LLEasAtt_representations/mustard_bagVectors_MM_NE_12_TVA_VTA_AVT.pkl' exists.
Working on: mustard
Video representation: bagVectors
Dataset mustard loaded successfully. Modality: T.
CLF: 32 <bound method BaseEstimator.get_params of MLPClassifier(max_iter=100000, random_state=1)>
Finished get_predictions
Saved ./eval_MLP_votes/votes_mustard_bagVectors_MM_NE_12_TVA_VTA_AVT.csv

Votes threshold: 6
Finished evaluate_labels
Finished classification_report
Saved ./eval_MLP_votes/preds_v6_mustard_bagVectors_MM_NE_12_TVA_VTA_AVT.csv
votes: 6, modality: T, acc: 0.6304347826086957, macro-f1: 0.6271258278145695, f1_0: 0.592, f1_1: 0.6622516556291391, variation: mustard_bagVectors_MM_NE_12_TVA_VTA_AVT

Results saved on: ./eval_MLP_votes/results_MM_NE.txt
